# W02D06 — Mini Project: Classification
**ML Engineer Journey | Phase 1 — ML Fundamentals**  
**Dataset**: Breast Cancer Wisconsin (sklearn built-in)  
**Models**: Linear Regression (baseline), Logistic Regression, Decision Tree  

---
Pipeline:
1. EDA — pahami data sebelum menyentuh model
2. Preprocessing — split + scale
3. Train models — 3 model + 1 dummy baseline
4. Diagnosis — learning curve dulu, baru cross_val_score
5. Kesimpulan — justifikasi berdasarkan grafik, bukan asumsi

## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('outputs/plots', exist_ok=True)

print('imports OK')

---
## 1. EDA — Exploratory Data Analysis

Sebelum menyentuh model, pahami dulu datanya:
- Berapa banyak sampel dan fitur?
- Apakah kelas seimbang? (class balance)
- Apa arti label 0 dan 1 di domain ini?

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

print('=' * 50)
print('BREAST CANCER WISCONSIN — EDA SUMMARY')
print('=' * 50)
print(f'Shape         : {X.shape}')           # (569, 30)
print(f'Features      : {X.shape[1]}')
print(f'Samples       : {X.shape[0]}')
print(f'Classes       : {data.target_names}')  # malignant=0, benign=1
print(f'Class balance : malignant={np.sum(y==0)}, benign={np.sum(y==1)}')
print(f'Positive rate : {np.mean(y==1):.1%}')
print()
print('Feature names (5 pertama):')
for name in data.feature_names[:5]:
    print(f'  - {name}')
print('  ...')
print('=' * 50)

In [ ]:
# Visualisasi distribusi kelas
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Malignant (0)', 'Benign (1)']
counts = [np.sum(y == 0), np.sum(y == 1)]
colors = ['#E24B4A', '#1D9E75']

bars = ax.bar(labels, counts, color=colors, edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', fontsize=11)

ax.set_title('Class Distribution — Breast Cancer Wisconsin', fontsize=13)
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig('outputs/plots/class_distribution.png', dpi=120)
plt.show()

# Catatan: dataset sedikit imbalanced — benign lebih banyak dari malignant
# Ini kenapa kita pakai stratify=y saat split nanti

---
## 2. Preprocessing — Split & Scale

Dua langkah wajib sebelum training:
- **`stratify=y`** — pastikan proporsi kelas sama di train dan test
- **`StandardScaler`** — fit HANYA di training data, transform di keduanya (hindari data leakage)

In [ ]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # WAJIB untuk dataset imbalanced
)

# Scale — fit hanya di training data
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)         # transform SAJA — tanpa fit ulang

print(f'Train : {X_train_sc.shape[0]} samples')
print(f'Test  : {X_test_sc.shape[0]} samples')
print(f'Train class balance : malignant={np.sum(y_train==0)}, benign={np.sum(y_train==1)}')
print(f'Test  class balance : malignant={np.sum(y_test==0)},  benign={np.sum(y_test==1)}')

---
## 3. Train Models

Empat model, dari paling naif ke paling proper:
- **DummyClassifier** — selalu prediksi kelas mayoritas. Ini batas bawah — kalau modelmu tidak bisa mengalahkan ini, ada yang salah fundamental.
- **LinearRegression** — bukan classifier asli. Output float, di-threshold 0.5. Dipakai sebagai numerical baseline.
- **LogisticRegression** — classifier proper dengan decision boundary linear.
- **DecisionTree** — non-linear, tanpa batasan depth (kita biarkan dulu untuk melihat overfitting).

In [ ]:
# Definisi model
models = {
    'Dummy (majority)' : DummyClassifier(strategy='most_frequent', random_state=42),
    'Linear Regression': LinearRegression(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'    : DecisionTreeClassifier(random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('=' * 60)
for name, model in models.items():
    model.fit(X_train_sc, y_train)

    # Handle LinearRegression yang output float — threshold 0.5
    if isinstance(model, LinearRegression):
        y_pred       = (model.predict(X_test_sc)  >= 0.5).astype(int)
        y_pred_train = (model.predict(X_train_sc) >= 0.5).astype(int)
        cv_scores = None
    else:
        y_pred       = model.predict(X_test_sc)
        y_pred_train = model.predict(X_train_sc)
        cv_scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='accuracy')

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc  = accuracy_score(y_test,  y_pred)

    results[name] = {
        'model'    : model,
        'train_acc': train_acc,
        'test_acc' : test_acc,
        'cv_mean'  : cv_scores.mean() if cv_scores is not None else None,
        'cv_std'   : cv_scores.std()  if cv_scores is not None else None,
    }

    cv_str = (f'CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}'
              if cv_scores is not None else 'CV: N/A')
    print(f'{name}')
    print(f'  Train acc : {train_acc:.4f} | Test acc : {test_acc:.4f} | {cv_str}')
    print()
print('=' * 60)

---
## 4. Diagnosis — Learning Curve (Baca ini SEBELUM menarik kesimpulan)

Urutan wajib:
1. Plot learning curve
2. Baca gap antara training score dan CV score
3. Lihat tren gap — mengecil atau stagnan?
4. **Baru** tarik kesimpulan dan tentukan aksi

Jangan skip langkah ini.

In [ ]:
classifier_names = ['Logistic Regression', 'Decision Tree']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_sizes = np.linspace(0.1, 1.0, 10)

for ax, name in zip(axes, classifier_names):
    model = models[name]

    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X_train_sc, y_train,
        train_sizes=train_sizes,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes_abs, train_mean, 'o-', color='#E24B4A', label='Training score', linewidth=2)
    ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std,
                    alpha=0.15, color='#E24B4A')
    ax.plot(train_sizes_abs, val_mean, 'o-', color='#1D9E75', label='CV score', linewidth=2)
    ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std,
                    alpha=0.15, color='#1D9E75')

    # Anotasi gap di titik terakhir — ini yang dibaca untuk diagnosis
    final_gap = train_mean[-1] - val_mean[-1]
    ax.annotate(f'gap={final_gap:.3f}',
                xy=(train_sizes_abs[-1], (train_mean[-1] + val_mean[-1]) / 2),
                xytext=(-60, 0), textcoords='offset points',
                fontsize=9, color='gray')

    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Training samples')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.75, 1.02)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves — W02D06 Classification Project', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/plots/learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Model Comparison — Bar Chart

In [ ]:
names     = list(results.keys())
test_accs = [results[n]['test_acc'] for n in names]
cv_means  = [results[n]['cv_mean'] if results[n]['cv_mean'] is not None else 0 for n in names]
cv_stds   = [results[n]['cv_std']  if results[n]['cv_std']  is not None else 0 for n in names]

x, width = np.arange(len(names)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width/2, test_accs, width, label='Test accuracy',
               color='#378ADD', edgecolor='white')
bars2 = ax.bar(x + width/2, cv_means, width, yerr=cv_stds, capsize=5,
               label='CV mean ± std', color='#1D9E75', edgecolor='white')

for bar in bars1:
    h = bar.get_height()
    if h > 0.81:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

# Dummy label manual (bar-nya di bawah ylim)
dummy_acc = results['Dummy (majority)']['test_acc']
ax.text(x[0], 0.802, f'{dummy_acc:.3f}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0.80, 1.02)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison — Test Accuracy & CV Score', fontsize=13)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/model_comparison.png', dpi=120)
plt.show()

---
## 6. Kesimpulan Final

Berdasarkan learning curve, model yang dipilih adalah **Logistic Regression**. Training score dan CV score konvergen dengan gap hanya 0.011, menunjukkan model berada di sweet spot — low bias, low variance — dengan CV score stabil di ~0.978.

Model yang paling bermasalah adalah **Decision Tree**: training score stagnan sempurna di 1.0 sementara CV score hanya 0.90–0.93 dengan gap 0.084 yang tidak mengecil seiring penambahan data. Ini adalah indikasi **high variance** — model menghafal training data dan gagal generalisasi. Solusi yang tepat adalah mengurangi kompleksitas model, misalnya dengan membatasi `max_depth`, bukan menambah data.

**Linear Regression** (0.956 test accuracy) performanya cukup baik untuk model yang bukan classifier, tapi tidak punya probabilitas output yang proper sehingga kurang ideal untuk kasus klasifikasi klinis seperti ini.

**DummyClassifier** (0.632) adalah batas bawah — semua model berhasil mengalahkannya.

In [ ]:
# Ringkasan tabel final
print('=' * 65)
print(f'{"Model":<22} {"Train":>8} {"Test":>8} {"CV Mean":>10} {"CV Std":>8}')
print('-' * 65)
for name, r in results.items():
    cv_m = f"{r['cv_mean']:.4f}" if r['cv_mean'] is not None else '   N/A'
    cv_s = f"{r['cv_std']:.4f}"  if r['cv_std']  is not None else '   N/A'
    print(f"{name:<22} {r['train_acc']:>8.4f} {r['test_acc']:>8.4f} {cv_m:>10} {cv_s:>8}")
print('=' * 65)
print()
print('Model terpilih : Logistic Regression')
print('Alasan         : gap terkecil (0.011), CV score tertinggi (0.978)')
print('Action item    : Decision Tree perlu max_depth untuk atasi high variance')

---
*W02D06 — Mini Project Classification | github.com/lolivampire/ML-Project*  
*Next: W03 — Feature Engineering & Evaluation Metrics*